In [ ]:
import torch
import sys
import os
from triton.testing import do_bench
from torch.utils.cpp_extension import load
import math

torch.set_float32_matmul_precision('high')

ext = load(
    name="attention_warp",
    sources=["/root/theCudaBender/attention_V4/export.cu"],
    extra_cuda_cflags=["-arch=sm_120", "-O3", "--use_fast_math"],
    extra_cflags=["-O3"],
    extra_ldflags=["-lcuda"],
    verbose=False,
)
BH, L_q, D, L_kv = ext.shape()
print(f"kernel config: BH={BH}, L_q={L_q}, D={D}, L_kv={L_kv}")


In [ ]:
class attention: 
  def __init__(self, B, H, D, L_q, L_kv, dtype=torch.bfloat16, device="cuda"): 
    # 4D layout for pytorch methods
    self.Q = torch.randn(B, H, L_q, D, dtype=dtype, device=device)
    self.K = torch.randn(B, H, L_kv, D, dtype=dtype, device=device)
    self.V = torch.randn(B, H, L_kv, D, dtype=dtype, device=device)
    # 3D layout for custom kernel: (BH, L, D), V column-major
    BH = B * H
    self.Q3 = self.Q.reshape(BH, L_q, D).contiguous()
    self.K3 = self.K.reshape(BH, L_kv, D).contiguous()
    self.V3 = torch.randn(BH, D, L_kv, dtype=dtype, device=device).transpose(1, 2)
    assert self.V3.stride(1) == 1
    self.scale = 1 / math.sqrt(D)
    self.B, self.H, self.D = B, H, D
    self.L_q, self.L_kv = L_q, L_kv
    self.flops = 4 * B * H * L_q * L_kv * D

  def base_attention(self): 
    S = (self.Q @ self.K.transpose(-2, -1)) * self.scale
    P = torch.nn.functional.softmax(S, dim=-1)
    return P @ self.V

  def sdpa_attention(self):
    return torch.nn.functional.scaled_dot_product_attention(
        self.Q, self.K, self.V, scale=self.scale
    )

  def cuda_kernel(self):
    return ext.attention(self.Q3, self.K3, self.V3)

  def _tflops(self, fn):
    eager_s = do_bench(fn, return_mode="median") * 1e-3
    compiled_s = do_bench(torch.compile(fn), return_mode="median") * 1e-3
    tf = self.flops * 1e-12
    return tf / eager_s, tf / compiled_s

  def bench_base(self):        return self._tflops(self.base_attention)
  def bench_sdpa(self):        return self._tflops(self.sdpa_attention)
  def bench_cuda_kernel(self): return self._tflops(self.cuda_kernel)


In [20]:
# Verify FlashAttention is available and will be used
from torch.nn.attention import SDPBackend, sdpa_kernel
print("flash_sdp enabled:", torch.backends.cuda.flash_sdp_enabled())
print("mem_efficient_sdp enabled:", torch.backends.cuda.mem_efficient_sdp_enabled())
print("math_sdp enabled:", torch.backends.cuda.math_sdp_enabled())


flash_sdp enabled: True
mem_efficient_sdp enabled: True
math_sdp enabled: True


In [ ]:
a = attention(B=4, H=8, D=128, L_q=8192, L_kv=4096)

base_eager,   base_compiled   = a.bench_base()
sdpa_eager,   sdpa_compiled   = a.bench_sdpa()
cuda_eager,   cuda_compiled   = a.bench_cuda_kernel()

print(f"base   eager:    {base_eager:.2f} TFLOPS")
print(f"base   compiled: {base_compiled:.2f} TFLOPS")
print(f"sdpa   eager:    {sdpa_eager:.2f} TFLOPS")
print(f"sdpa   compiled: {sdpa_compiled:.2f} TFLOPS")
print(f"kernel eager:    {cuda_eager:.2f} TFLOPS")
print(f"kernel compiled: {cuda_compiled:.2f} TFLOPS")
